# Comparación jerárquica de modelos entrenados

In [ ]:

%matplotlib inline

In [ ]:
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from loguru import logger

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

from src.data.dataset import reconstruct_test_data
from src.evaluation import build_predictions_report
from src.evaluation.point import bias, rmse, wape
from src.evaluation.scaled import compute_naive_scales

artifact_paths = sorted(Path("../artifacts/models").glob("*_artifact.pkl"))

In [ ]:
comparative = []
all_model_results = []

for artifact_path in artifact_paths:

    artifact = joblib.load(artifact_path)

    grainly = "daily" if "daily" in artifact_path.name else "weekly"

    comparative.append({
        "level_label": artifact["level_label"],
        "level_id": artifact["level_id"],
        "grainly": grainly,
        "winner_family": artifact["winner_family"],
        "wape_test": artifact["wape_test"],
        "wrmsse_test": artifact["wrmsse_test"],
        "wape_valid": artifact["wape_valid"],
        "wrmsse_valid": artifact["wrmsse_valid"],
    })

    # Guarda el detalle de TODOS los modelos evaluados en este nivel
    for model_result in artifact["model_results"]:
        all_model_results.append({
            "level_label": artifact["level_label"],
            "level_id": artifact["level_id"],
            "grainly": grainly,
            **model_result,
        })

df_comparative = pd.DataFrame(comparative)
df_all_models = pd.DataFrame(all_model_results)

# Promedio de performance por modelo a nivel general (todos los niveles/grains)
df_model_avg = (
    df_all_models
    .groupby("model")[["wape", "wrmsse", "mae", "rmse", "smape", "bias", "mase"]]
    .mean()
    .sort_values("wrmsse")
)

In [ ]:
is_tuned_variant = df_all_models["model"].str.contains(
    r"\(bench, selected features\)|\(final, Optuna\)", regex=True
)
df_fair = df_all_models[~is_tuned_variant].copy()

df_tuned_variants = df_all_models[is_tuned_variant].copy()

In [ ]:
df_tuned_avg = (
    df_tuned_variants
    .groupby(["model","grainly"])[["wape", "wrmsse"]]
    .mean()
)

df_tuned_avg["n_levels"] = df_tuned_variants.groupby(["model","grainly"]).size()
df_tuned_avg = df_tuned_avg.sort_values("wape").reset_index()

df_tuned_avg = df_tuned_avg.query("grainly == 'daily'")

df_tuned_avg.nsmallest(10, "wape")

In [ ]:
df_fair_avg = (
    df_fair
    .groupby(["model","grainly"])[["wape", "wrmsse"]]
    .mean()
)

df_fair_avg["n_levels"] = df_fair.groupby(["model","grainly"]).size()
df_fair_avg = df_fair_avg.sort_values("wape").reset_index()

df_fair_avg = df_fair_avg.query("grainly == 'daily'")

df_fair_avg.nsmallest(10, "wape")

**Nota:** `SARIMA` en `df_family_avg` va a salir con wape/wrmsse en el orden de millones -- es un artifact roto (diverge), no un resultado real. Excluir de cualquier tabla/gráfico para el informe hasta confirmar la causa, no solo recortar el eje.

El promedio de `df_family_avg` es el piso de cada familia ML (sin feature selection ni tuning). Para ver si esa foto global esconde a una familia que gana fuerte en algunos niveles y pierde en otros, se abre por nivel debajo.

In [ ]:
ml_families = ["xgboost", "lightgbm", "histgb", "catboost", "ridge"]
bench_names = [f"{fam} (bench)" for fam in ml_families]

df_bench_ml = df_fair[df_fair["model"].isin(bench_names)].copy()
df_bench_ml["family"] = df_bench_ml["model"].str.replace(r" \(bench\)$", "", regex=True)

pivot_bench_wape = df_bench_ml.pivot_table(index="level_label", columns="family", values="wape")
pivot_bench_wape["winner_family"] = df_comparative.set_index("level_label")["winner_family"]

pivot_bench_wape.style.highlight_min(subset=ml_families, axis=1, color="#c6efce")

## 0. Comparativo

In [ ]:
artifact_paths = sorted(Path("../artifacts/models").glob("*_artifact.pkl"))

comparative = []

for artifact_path in artifact_paths:

    artifact = joblib.load(artifact_path)

    grainly = "daily" if "daily" in artifact_path.name else "weekly"

    comparative.append({
        "level_label": artifact["level_label"],
        "level_id": artifact["level_id"],
        "grainly": grainly,
        "winner_family": artifact["winner_family"],
        "wape_test": artifact["wape_test"],
        "wrmsse_test": artifact["wrmsse_test"],
        "wape_valid": artifact["wape_valid"],
        "wrmsse_valid": artifact["wrmsse_valid"],
    })

df_comparative = pd.DataFrame(comparative)

df_comparative

In [ ]:
plt.figure(figsize=(9, 4))

sns.barplot(
    data=df_comparative,
    x="level_label",
    y="wape_test",
    hue="grainly"
)

for container in plt.gca().containers:
    for bar in container:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.2f}",
            ha="center",
            va="bottom",
            size=9
        )

plt.xticks(rotation=45, ha="right")

plt.yticks(np.arange(0, 0.13, 0.025))

plt.title("Comparación del WAPE ent test según nivel de agregación y frecuencia", fontsize=12, fontweight="bold", pad=15, loc="left")

plt.legend(
    title="Frecuencia",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))

sns.barplot(
    data=df_comparative,
    x="level_label",
    y="wrmsse_test",
    hue="grainly"
)

for container in plt.gca().containers:
    for bar in container:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.2f}",
            ha="center",
            va="bottom",
            size=9
        )

plt.xticks(rotation=45, ha="right")

plt.yticks(np.arange(0, 1.25, 0.25))

plt.title("Comparación del WRMSSE de test según nivel de agregación y frecuencia", fontsize=12, fontweight="bold", pad=15, loc="left")

plt.legend(
    title="Frecuencia",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
df_metrics_long = df_comparative.melt(
    id_vars=["level_label", "grainly"],
    value_vars=["wape_test", "wape_valid", "wrmsse_test", "wrmsse_valid"],
    var_name="metric_conjunto",
    value_name="valor",
)

df_metrics_long[["metric", "conjunto"]] = df_metrics_long["metric_conjunto"].str.split("_", expand=True)

df_metrics_long.drop(columns=["metric_conjunto"], inplace=True)

df_metrics_long = df_metrics_long.set_index('level_label')

df_metrics_long.head()

In [ ]:
df_metrics_long['metric'].unique()

In [ ]:
df_metrics_long.query("metric == 'wrmsse'")

In [ ]:
df_wrmsse_daily = df_metrics_long.query("metric == 'wrmsse' and grainly == 'daily'")

plt.figure(figsize=(9, 4))
sns.barplot(
    data=df_wrmsse_daily,
    x="level_label",
    y="valor",
    hue="conjunto"
)

for container in plt.gca().containers:
    for bar in container:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.2f}",
            ha="center",
            va="bottom",
            size=9
        )

plt.xticks(rotation=45, ha="right")

plt.yticks(np.arange(0, 2.5, 0.25))

plt.ylabel("WRMSSE")

plt.title("Comparación del WRMSSE de conjuntos test vs valid", fontsize=12, fontweight="bold", pad=15, loc="left")

plt.legend(
    title="Frecuencia",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
df_wape_daily = df_metrics_long.query("metric == 'wape' and grainly == 'daily'")

plt.figure(figsize=(9, 4))
sns.barplot(
    data=df_wape_daily,
    x="level_label",
    y="valor",
    hue="conjunto"
)

for container in plt.gca().containers:
    for bar in container:
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.2f}",
            ha="center",
            va="bottom",
            size=9
        )

plt.xticks(rotation=45, ha="right")

plt.ylabel("WAPE")

plt.yticks(np.arange(0, .6, 0.1))

plt.title("Comparación del WAPE de conjuntos test vs valid", fontsize=12, fontweight="bold", pad=15, loc="left")

plt.legend(
    title="Conjunto",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

## 1. Carga de artifacts, reconstrucción de test y predicciones

## 1.1 Carga

In [ ]:
DIMS = ["dept_id", "cat_id", "store_id", "state_id"]

def load_level(artifact_path: Path) -> dict:
    '''Carga un artifact, reconstruye su X_test/train/test y arma el detalle de
    predicciones (df_pred) con las columnas de dimension (dept_id/store_id/...)
    pegadas desde X_test, para poder reagregar a cualquier nivel mas agregado.'''
    artifact = joblib.load(artifact_path)
    model = artifact["model"]

    X_test, y_test, train, test = reconstruct_test_data(artifact)
    y_pred = model.predict(X_test)

    metrics, df_pred = build_predictions_report(
        train, test, y_test, y_pred, target_col=artifact["target"],
    )
    dims_available = [d for d in DIMS if d in X_test.columns]
    df_pred = df_pred.join(X_test[dims_available])

    return {
        "level_label": artifact["level_label"],
        "level_id": artifact["level_id"],
        "level": artifact["level"],
        "model_class": type(model).__name__,
        "n_series": df_pred["series_id"].nunique(),
        "metrics": metrics,
        "df_pred": df_pred,
        "train": train,
        "dims_available": dims_available,
    }


artifact_paths = sorted(Path("../artifacts/models").glob("*_artifact.pkl"))
logger.info("Artifacts encontrados: {}", [p.name for p in artifact_paths])

LEVELS = {}
for path in artifact_paths:
    result = load_level(path)
    LEVELS[result["level_label"]] = result
    logger.info(
        "{} | {} series | modelo={} | WAPE test={:.2%} | WRMSSE test={:.3f}",
        result["level"], result["n_series"], result["model_class"],
        result["metrics"]["wape"], result["metrics"]["wrmsse"],
    )


## 2. Resumen de cada modelo en su propio nivel nativo

In [ ]:
summary_rows = []
for level_id, res in sorted(LEVELS.items()):
    m = res["metrics"]
    summary_rows.append({
        "level_label": res["level_label"],
        "level": res["level"],
        "n_series": res["n_series"],
        "modelo": res["model_class"],
        "wape": m["wape"],
        "wrmsse": m["wrmsse"],
        "bias": m["bias"],
        "rmse": m["rmse"],
    })

df_summary = pd.DataFrame(summary_rows).set_index("level_label")
df_summary.style.format({"wape": "{:.2%}", "wrmsse": "{:.3f}", "bias": "{:.2%}", "rmse": "{:,.1f}"})


## 3. Comparación al nivel TOTAL

Se agregan (`groupby("date").sum()`) las predicciones de cada nivel a la serie
diaria total y se comparan contra el mismo real (`sales`), que por construcción es
idéntico en todos los niveles. También se arma una combinación simple: el
promedio de las 4 predicciones totales, como primer paso hacia una reconciliación
jerárquica más formal (bottom-up ya está acá; top-down por proporciones históricas
o MinT/optimal reconciliation -- p.ej. con `hierarchicalforecast` -- quedan como
extensión natural).

In [ ]:
def aggregate_to_total(df_pred: pd.DataFrame) -> pd.DataFrame:
    agg = df_pred.groupby("date").agg(actual=("sales", "sum"), y_pred=("y_pred", "sum"))
    return agg


def to_weekly(agg: pd.DataFrame) -> pd.DataFrame:
    """Reagrega una serie diaria a semanal (lunes-domingo, fecha = lunes de esa
    semana, igual que el date_trunc('week', ...) usado al construir los datasets
    weekly -- ver src/data/base_query.py) para que sea comparable con L1, que
    solo tiene artifact weekly."""
    week_start = agg.index - pd.to_timedelta(agg.index.dayofweek, unit="D")
    return agg.groupby(week_start).sum()


# Solo se agregan al total los niveles que particionan por completo la serie total
# (item/item_store, 10-12, están filtrados a un subconjunto dept/store y no sumarían al total real).
VALID_LEVELS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# LEVELS está keyeado por level_label ("level_04_daily", "level_04_weekly", ...),
# no por level_id: varios niveles (3, 5-9) tienen artifacts en ambos grains -- nos
# quedamos con el nativo daily de cada nivel (más fino) y sólo caemos a weekly
# cuando no hay daily entrenado (caso de L1 hoy).
GRAIN_PRIORITY = {"daily": 0, "weekly": 1}

def grain_of(res: dict) -> str:
    return res["level_label"].rsplit("_", 1)[-1]

levels_by_id: dict[int, dict] = {}
for res in LEVELS.values():
    level_id = res["level_id"]
    current = levels_by_id.get(level_id)
    if current is None or GRAIN_PRIORITY[grain_of(res)] < GRAIN_PRIORITY[grain_of(current)]:
        levels_by_id[level_id] = res

# L1 solo existe en weekly: todo se reagrega a semanal (lunes de cada semana)
# para poder comparar contra L1 en la misma base -- sumar los diarios a nivel de
# semana no pierde nada (es la misma cuenta agregada distinto), pero unirlos sin
# reagregar sí: las fechas diarias y semanales no calzan al pegar columnas por
# índice (cada fecha semanal solo matchea el día puntual, no la semana entera).
total_aggs = {
    level_id: to_weekly(aggregate_to_total(res["df_pred"])) if grain_of(res) == "daily"
    else aggregate_to_total(res["df_pred"])
    for level_id, res in levels_by_id.items()
    if level_id in VALID_LEVELS
}

df_total = pd.DataFrame({"actual": next(iter(total_aggs.values()))["actual"]})
for level_id, agg in sorted(total_aggs.items()):
    df_total[f"L{level_id}"] = agg["y_pred"]

# combinación naive: promedio de las predicciones de todos los niveles disponibles
pred_cols = [c for c in df_total.columns if c != "actual"]
#df_total["Promedio de niveles"] = df_total[pred_cols].mean(axis=1)

df_total.round(0).head()

In [ ]:
# Escala WRMSSE del total: MSE del naive de 1 día in-sample sobre el train del
# nivel L1 (la serie total ya es una sola serie -- no hay ponderación entre grupos).
train_total = levels_by_id[1]["train"][["date", "sales"]].copy()
train_total["series_id"] = "TOTAL"
scale_total = compute_naive_scales(train_total, group_col="series_id", target_col="sales")["TOTAL"]

rows = []
for col in [c for c in df_total.columns if c != "actual"]:
    y_true, y_pred = df_total["actual"], df_total[col]
    rows.append({
        "enfoque": col,
        "wape": wape(y_true, y_pred),
        "bias": bias(y_true, y_pred),
        "rmse": rmse(y_true, y_pred),
        "wrmsse": rmse(y_true, y_pred) / np.sqrt(scale_total),
    })

df_total_metrics = pd.DataFrame(rows).sort_values("wape").reset_index(drop=True)
df_total_metrics

In [ ]:
import matplotlib.pyplot as plt

df_plot = df_total_metrics.set_index("enfoque")

metrics = {"wape": ("WAPE", "%"), "wrmsse": ("WRMSSE", "num")}
colors = {"wape": "#4C72B0", "bias": "#DD8452", "wrmsse": "#55A868"}

for metric, (title, fmt_type) in metrics.items():
    sort_type = False if metric in ["wape", "wrmsse"] else False
    data = df_plot[metric].sort_values(ascending=sort_type)

    fig, ax = plt.subplots(figsize=(6, 4))
    data.plot(kind="barh", ax=ax, color=colors[metric], edgecolor="none")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False)
    ax.set_xticks([])

    for i, v in enumerate(data):
        label = f"{v:.2%}" if fmt_type == "%" else f"{v:.3f}"
        ax.text(v, i, f" {label}", va="center", fontsize=11)

    plt.tight_layout()
    plt.grid(False)
    plt.show()

In [ ]:
df_plot = df_total[['actual', 'L1', 'L2','L2']].copy()

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(df_plot.index, df_plot["actual"], label="Real", color="black",
        linewidth=2.5, marker="o", markersize=4, zorder=3)

styles = [
    {"color": "#E4572E", "linestyle": "--", "marker": "s"},
    {"color": "#4C72B0", "linestyle": "-.", "marker": "^"},
    {"color": "#55A868", "linestyle": ":", "marker": "D"},
]

for col, style in zip([c for c in df_plot.columns if c != "actual"], styles):
    ax.plot(df_plot.index, df_plot[col], label=col, alpha=0.8,
            linewidth=1.8, markersize=4, zorder=2, **style)

ax.set_title("Total de ventas diarias: real vs. directo (L1) vs. bottom-up (L5/L8)", loc="left")
ax.set_xlabel("")
ax.set_xticks(df_plot.index)
ax.legend(loc="upper left", fontsize=9, frameon=False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.grid(alpha=0.3)
plt.ylim(0)
plt.show()

## Conclusiones rápidas

- Revisar `df_total_metrics`: si "Promedio de niveles" queda por debajo del mejor
  enfoque individual, hay señal de que combinar información de distintos niveles
  de agregación reduce error -- la motivación estándar detrás de la reconciliación
  jerárquica (bottom-up, top-down, MinT).
- `df_dept_compare` / `df_store_compare` (columna `gana_bottomup`) muestran, serie
  por serie, si conviene más el modelo especializado en ese nivel o la suma desde
  el nivel más fino -- normalmente depende de cuánta señal específica del grupo
  (dept/store) exista vs. cuánta información se gana de tener series individuales
  más finas para el modelo de L9.
- Extensión natural: formalizar esto con reconciliación óptima (p.ej.
  `hierarchicalforecast` de Nixtla, que ya cubre bottom-up/top-down/MinT sobre esta
  misma estructura de niveles).
